<a href="https://colab.research.google.com/github/mashac136/RealTimeMachineLearning/blob/main/hw5_vit_swin_cifar100_Marly_Ashac!.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1: CIFAR-100 loading & preprocessing

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import time

# CPU fallback: automatically use GPU if available - otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CIFAR-100 normalization stats
CIFAR100_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR100_STD = (0.2673, 0.2564, 0.2762)

# basic augmentation (random crop + flip) to help generalization
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),      # random crop with padding - standard for CIFAR
    transforms.RandomHorizontalFlip(),          # random horizontal flip
    transforms.ToTensor(),                      # convert PIL image to tensor
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),  # normalize using CIFAR-100 stats
])

# no augmentation - just convert + normalize
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

# Load CIFAR-100 train and test sets
train_dataset = torchvision.datasets.CIFAR100(
    root=DRIVE_DATA_PATH, train=True, download=True, transform=train_transform
)
test_dataset = torchvision.datasets.CIFAR100(
    root=DRIVE_DATA_PATH, train=False, download=True, transform=test_transform
)

# batch size of 64
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

# confirm shapes and number of classes
print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of classes: {len(train_dataset.classes)}")

# grab one batch to confirm shapes are correct (32x32 RGB images)
sample_images, sample_labels = next(iter(train_loader))
print(f"Batch image shape: {sample_images.shape}")
print(f"Batch label shape: {sample_labels.shape}")

Using device: cuda
Train samples: 50000
Test samples: 10000
Number of classes: 100
Batch image shape: torch.Size([64, 3, 32, 32])
Batch label shape: torch.Size([64])


In [1]:
# 1b: mount google drive & cache dataset - this is for not having to re-load dataset every time

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DATA_PATH = "/content/drive/MyDrive/cifar100_cache"
os.makedirs(DRIVE_DATA_PATH, exist_ok=True)

print(f"Drive mounted. Dataset will be cached at: {DRIVE_DATA_PATH}")

Mounted at /content/drive
Drive mounted. Dataset will be cached at: /content/drive/MyDrive/cifar100_cache


In [ ]:
# 2: vision transformer (ViT) architecture - scratch

import torch
import torch.nn as nn

class PatchEmbedding(nn.Module):
    """
    Splits the image into fixed-size patches and linearly projects
    each patch into an embedding vector (like word embeddings, but for image patches).
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, embed_dim=256):
        super().__init__()
        self.num_patches = (img_size // patch_size) ** 2  # e.g., (32/4)^2 = 64 patches

        # Conv2d with kernel_size=stride=patch_size does patch splitting + projection in one step
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        # x shape: [batch, channels, height, width]
        x = self.proj(x)               # to [batch, embed_dim, H/patch, W/patch]
        x = x.flatten(2)                # to [batch, embed_dim, num_patches]
        x = x.transpose(1, 2)           # to [batch, num_patches, embed_dim]
        return x


class TransformerEncoderBlock(nn.Module):
    """
    Standard transformer encoder block: multi-head self-attention + MLP,
    each with a residual connection and LayerNorm (pre-norm style).
    """
    def __init__(self, embed_dim, num_heads, mlp_ratio=4, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)

        # MLP hidden dim = 4x embed_dim
        mlp_hidden_dim = embed_dim * mlp_ratio
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # LayerNorm before attention - residual connection after
        normed = self.norm1(x)
        attn_out, _ = self.attn(normed, normed, normed)  # self-attention: query=key=value=x
        x = x + attn_out                                  # residual connection

        # LayerNorm before MLP, residual connection after
        x = x + self.mlp(self.norm2(x))
        return x


class VisionTransformer(nn.Module):
    """
    Full ViT: patch embedding -> class token + positional embedding ->
    transformer encoder blocks -> classification head.
    Configurable so we can sweep patch_size, embed_dim, depth, and num_heads.
    """
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=100,
                 embed_dim=256, depth=4, num_heads=4, mlp_ratio=4, dropout=0.1):
        super().__init__()

        # patch embedding layer
        self.patch_embed = PatchEmbedding(img_size, patch_size, in_channels, embed_dim)
        num_patches = self.patch_embed.num_patches

        #[CLS] token - prepended to patch sequence & used for final classification
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))

        # learnable positional embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(dropout)

        # stack of transformer encoder blocks (depth = number of blocks)
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_ratio, dropout)
            for _ in range(depth)
        ])

        # final LayerNorm before classification head
        self.norm = nn.LayerNorm(embed_dim)

        # classification head: maps cls token embedding to class logits
        self.head = nn.Linear(embed_dim, num_classes)

        # initialize cls_token and pos_embed with small random values
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)

    def forward(self, x):
        batch_size = x.shape[0]

        x = self.patch_embed(x)  # to [batch, num_patches, embed_dim]

        # cls token to the patch sequence
        cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # to [batch, 1, embed_dim]
        x = torch.cat((cls_tokens, x), dim=1)                    # -to [batch, num_patches+1, embed_dim]

        # add positional embeddings
        x = x + self.pos_embed
        x = self.pos_drop(x)

        # pass through transformer encoder blocks
        for block in self.blocks:
            x = block(x)

        x = self.norm(x)

        # use only the cls tokens final embedding for classification
        cls_output = x[:, 0]
        logits = self.head(cls_output)
        return logits


#  build a small ViT and verify output shape
test_model = VisionTransformer(patch_size=4, embed_dim=256, depth=4, num_heads=4).to(device)
test_input = torch.randn(2, 3, 32, 32).to(device)  # batch of 2 dummy images
test_output = test_model(test_input)
print(f"Test output shape: {test_output.shape}")  # expect [2, 100] to batch of 2, 100 classes
del test_model, test_input, test_output  # clean up test objects
torch.cuda.empty_cache()

Test output shape: torch.Size([2, 100])


In [ ]:
# 3: ViT configuration definitions

# 4 configs - each isolating one architectural variable vs. Config A (baseline)

vit_configs = {
    "Config_A_baseline": {
        "patch_size": 4, "embed_dim": 256, "depth": 4, "num_heads": 4
    },
    "Config_B_wider": {
        "patch_size": 4, "embed_dim": 512, "depth": 4, "num_heads": 8
    },
    "Config_C_larger_patch": {
        "patch_size": 8, "embed_dim": 256, "depth": 4, "num_heads": 4
    },
    "Config_D_deeper": {
        "patch_size": 4, "embed_dim": 256, "depth": 8, "num_heads": 4
    },
}

# print configs to confirm before training
for name, cfg in vit_configs.items():
    print(f"{name}: {cfg}")

Config_A_baseline: {'patch_size': 4, 'embed_dim': 256, 'depth': 4, 'num_heads': 4}
Config_B_wider: {'patch_size': 4, 'embed_dim': 512, 'depth': 4, 'num_heads': 8}
Config_C_larger_patch: {'patch_size': 8, 'embed_dim': 256, 'depth': 4, 'num_heads': 4}
Config_D_deeper: {'patch_size': 4, 'embed_dim': 256, 'depth': 8, 'num_heads': 4}


In [ ]:
# 4: training loop

import torch.optim as optim

def train_model(model, train_loader, test_loader, num_epochs=10, lr=0.001, model_name="model"):
    """
    Trains a model for num_epochs, tracking loss, test accuracy, and timing per epoch.
    Returns a dict of results: per-epoch train loss, test accuracy, and total training time.
    """
    model = model.to(device)

    # Adam optimizer, lr=0.001
    optimizer = optim.Adam(model.parameters(), lr=lr)

    #  cross-entropy loss for multi-class classification
    criterion = nn.CrossEntropyLoss()

    # track results across epochs
    train_losses = []
    test_accuracies = []
    epoch_times = []

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # training phase
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()          # clear gradients from previous step
            outputs = model(images)         # forward pass
            loss = criterion(outputs, labels)
            loss.backward()                 # backprop
            optimizer.step()                # update weights

            running_loss += loss.item() * images.size(0)

        avg_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        # evaluation phase
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():  # no gradient tracking needed for evaluation
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total
        test_accuracies.append(test_accuracy)

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Test Acc: {test_accuracy:.2f}% | "
              f"Time: {epoch_time:.1f}s")

    total_training_time = time.time() - total_start_time

    results = {
        "model_name": model_name,
        "train_losses": train_losses,
        "test_accuracies": test_accuracies,
        "final_test_accuracy": test_accuracies[-1],
        "epoch_times": epoch_times,
        "avg_epoch_time": sum(epoch_times) / len(epoch_times),
        "total_training_time": total_training_time,
        "num_params": sum(p.numel() for p in model.parameters()),
    }

    print(f"\nFinal Test Accuracy: {results['final_test_accuracy']:.2f}%")
    print(f"Total Training Time: {total_training_time:.1f}s ({total_training_time/60:.1f} min)")
    print(f"Number of Parameters: {results['num_params']:,}")

    return results

In [ ]:

# 5: train ViT baseline

# ViT using (A) hyperparameters
model_A = VisionTransformer(
    img_size=32,
    patch_size=vit_configs["Config_A_baseline"]["patch_size"],
    in_channels=3,
    num_classes=100,
    embed_dim=vit_configs["Config_A_baseline"]["embed_dim"],
    depth=vit_configs["Config_A_baseline"]["depth"],
    num_heads=vit_configs["Config_A_baseline"]["num_heads"],
    mlp_ratio=4,   # MLP hidden dim = 4x embed_dim
)

# train for 10 epochs, batch size 64 (already set in train_loader/test_loader), lr=0.001
results_A = train_model(
    model_A, train_loader, test_loader,
    num_epochs=10, lr=0.001, model_name="Config_A_baseline"
)


Training: Config_A_baseline
Epoch 1/10 | Train Loss: 3.9684 | Test Acc: 11.84% | Time: 39.5s
Epoch 2/10 | Train Loss: 3.6038 | Test Acc: 16.75% | Time: 36.3s
Epoch 3/10 | Train Loss: 3.3935 | Test Acc: 19.97% | Time: 37.2s
Epoch 4/10 | Train Loss: 3.2220 | Test Acc: 22.99% | Time: 36.2s
Epoch 5/10 | Train Loss: 3.0949 | Test Acc: 25.60% | Time: 37.2s
Epoch 6/10 | Train Loss: 2.9740 | Test Acc: 28.02% | Time: 36.2s
Epoch 7/10 | Train Loss: 2.8776 | Test Acc: 28.74% | Time: 37.3s
Epoch 8/10 | Train Loss: 2.7867 | Test Acc: 31.63% | Time: 36.1s
Epoch 9/10 | Train Loss: 2.7106 | Test Acc: 32.72% | Time: 37.3s
Epoch 10/10 | Train Loss: 2.6315 | Test Acc: 34.23% | Time: 36.2s

Final Test Accuracy: 34.23%
Total Training Time: 369.4s (6.2 min)
Number of Parameters: 3,214,692


In [ ]:
# 6: Train ViT (wider)

# instantiate ViT using (B) hyperparameters
model_B = VisionTransformer(
    img_size=32,
    patch_size=vit_configs["Config_B_wider"]["patch_size"],
    in_channels=3,
    num_classes=100,
    embed_dim=vit_configs["Config_B_wider"]["embed_dim"],
    depth=vit_configs["Config_B_wider"]["depth"],
    num_heads=vit_configs["Config_B_wider"]["num_heads"],
    mlp_ratio=4,   # MLP hidden dim = 4x embed_dim, as specified in the assignment
)

# train for 10 epochs - batch size 64 - lr=0.001
results_B = train_model(
    model_B, train_loader, test_loader,
    num_epochs=10, lr=0.001, model_name="Config_B_wider"
)


Training: Config_B_wider
Epoch 1/10 | Train Loss: 4.1579 | Test Acc: 8.47% | Time: 109.1s
Epoch 2/10 | Train Loss: 4.0217 | Test Acc: 6.82% | Time: 108.4s
Epoch 3/10 | Train Loss: 4.0760 | Test Acc: 7.01% | Time: 107.6s
Epoch 4/10 | Train Loss: 4.0996 | Test Acc: 7.72% | Time: 107.2s
Epoch 5/10 | Train Loss: 4.0468 | Test Acc: 8.73% | Time: 106.0s
Epoch 6/10 | Train Loss: 4.0702 | Test Acc: 6.83% | Time: 105.7s
Epoch 7/10 | Train Loss: 4.1104 | Test Acc: 7.22% | Time: 104.1s
Epoch 8/10 | Train Loss: 4.1041 | Test Acc: 7.97% | Time: 103.6s
Epoch 9/10 | Train Loss: 4.1070 | Test Acc: 7.48% | Time: 103.9s
Epoch 10/10 | Train Loss: 4.1177 | Test Acc: 7.17% | Time: 102.9s

Final Test Accuracy: 7.17%
Total Training Time: 1058.4s (17.6 min)
Number of Parameters: 12,720,740


In [ ]:
#7: train ViT (larger patch)


# instantiate ViT (C) hyperparameters
model_C = VisionTransformer(
    img_size=32,
    patch_size=vit_configs["Config_C_larger_patch"]["patch_size"],
    in_channels=3,
    num_classes=100,
    embed_dim=vit_configs["Config_C_larger_patch"]["embed_dim"],
    depth=vit_configs["Config_C_larger_patch"]["depth"],
    num_heads=vit_configs["Config_C_larger_patch"]["num_heads"],
    mlp_ratio=4,   # MLP hidden dim = 4x embed_dim, as specified in the assignment
)

# train for 10 epochs, batch size 64 - lr=0.001
results_C = train_model(
    model_C, train_loader, test_loader,
    num_epochs=10, lr=0.001, model_name="Config_C_larger_patch"
)


Training: Config_C_larger_patch
Epoch 1/10 | Train Loss: 4.0760 | Test Acc: 9.78% | Time: 30.1s
Epoch 2/10 | Train Loss: 3.8720 | Test Acc: 10.03% | Time: 29.6s
Epoch 3/10 | Train Loss: 3.8232 | Test Acc: 11.68% | Time: 30.3s
Epoch 4/10 | Train Loss: 3.8105 | Test Acc: 10.55% | Time: 29.3s
Epoch 5/10 | Train Loss: 3.8012 | Test Acc: 12.57% | Time: 29.9s
Epoch 6/10 | Train Loss: 3.7606 | Test Acc: 12.15% | Time: 29.9s
Epoch 7/10 | Train Loss: 3.7434 | Test Acc: 11.93% | Time: 30.3s
Epoch 8/10 | Train Loss: 3.7145 | Test Acc: 13.76% | Time: 29.5s
Epoch 9/10 | Train Loss: 3.7005 | Test Acc: 12.54% | Time: 29.6s
Epoch 10/10 | Train Loss: 3.7600 | Test Acc: 11.74% | Time: 30.5s

Final Test Accuracy: 11.74%
Total Training Time: 298.9s (5.0 min)
Number of Parameters: 3,239,268


In [ ]:
# 8: train ViT (deeper)

# instantiate ViT using (D) hyperparameters
model_D = VisionTransformer(
    img_size=32,
    patch_size=vit_configs["Config_D_deeper"]["patch_size"],
    in_channels=3,
    num_classes=100,
    embed_dim=vit_configs["Config_D_deeper"]["embed_dim"],
    depth=vit_configs["Config_D_deeper"]["depth"],
    num_heads=vit_configs["Config_D_deeper"]["num_heads"],
    mlp_ratio=4,   # MLP hidden dim = 4x embed_dim, as specified in the assignment
)

# train for 10 epochs, batch size 64 - lr=0.001
results_D = train_model(
    model_D, train_loader, test_loader,
    num_epochs=10, lr=0.001, model_name="Config_D_deeper"
)


Training: Config_D_deeper
Epoch 1/10 | Train Loss: 4.0528 | Test Acc: 10.93% | Time: 71.1s
Epoch 2/10 | Train Loss: 3.7674 | Test Acc: 11.00% | Time: 71.4s
Epoch 3/10 | Train Loss: 3.7193 | Test Acc: 12.06% | Time: 69.7s
Epoch 4/10 | Train Loss: 3.6774 | Test Acc: 13.80% | Time: 69.9s
Epoch 5/10 | Train Loss: 3.6830 | Test Acc: 13.51% | Time: 69.9s
Epoch 6/10 | Train Loss: 3.6666 | Test Acc: 13.95% | Time: 69.7s
Epoch 7/10 | Train Loss: 3.6232 | Test Acc: 15.69% | Time: 69.6s
Epoch 8/10 | Train Loss: 3.6335 | Test Acc: 13.24% | Time: 69.7s
Epoch 9/10 | Train Loss: 3.6447 | Test Acc: 14.65% | Time: 70.5s
Epoch 10/10 | Train Loss: 3.6075 | Test Acc: 14.33% | Time: 69.9s

Final Test Accuracy: 14.33%
Total Training Time: 701.3s (11.7 min)
Number of Parameters: 6,373,732


In [ ]:
# 9: FLOPs & parameter count for all ViT

# install torchinfo if not available
!pip install -q torchinfo

from torchinfo import summary

# rebuild each model fresh
flops_results = {}

for config_name, cfg in vit_configs.items():
    model = VisionTransformer(
        img_size=32,
        patch_size=cfg["patch_size"],
        in_channels=3,
        num_classes=100,
        embed_dim=cfg["embed_dim"],
        depth=cfg["depth"],
        num_heads=cfg["num_heads"],
        mlp_ratio=4,
    ).to(device)

    # torchinfo traces a forward pass with a dummy input to compute FLOPs (mult-adds) and params
    model_summary = summary(
        model,
        input_size=(1, 3, 32, 32),  # batch size 1 for per-sample FLOPs
        verbose=0,
        col_names=["num_params", "mult_adds"],
    )

    total_flops = model_summary.total_mult_adds  # multiply-add operations - standard FLOPs proxy
    total_params = model_summary.total_params

    flops_results[config_name] = {
        "total_params": total_params,
        "flops_per_forward_pass": total_flops,
    }

    print(f"{config_name}:")
    print(f"  Params: {total_params:,}")
    print(f"  FLOPs (mult-adds) per forward pass: {total_flops:,}")
    print()

    del model  # free GPU memory
    torch.cuda.empty_cache()

Config_A_baseline:
  Params: 3,214,692
  FLOPs (mult-adds) per forward pass: 2,935,396

Config_B_wider:
  Params: 12,720,740
  FLOPs (mult-adds) per forward pass: 10,064,996

Config_C_larger_patch:
  Params: 3,239,268
  FLOPs (mult-adds) per forward pass: 2,923,108

Config_D_deeper:
  Params: 6,373,732
  FLOPs (mult-adds) per forward pass: 5,041,764



In [ ]:
# 10: ResNet-18 baseline (trained from scratch)

import torchvision.models as models

# load ResNet-18 architecture wihtout pretrained weights
resnet18 = models.resnet18(weights=None)

# adapt the final fully-connected layer for CIFAR-100's 100 classes
resnet18.fc = nn.Linear(resnet18.fc.in_features, 100)

resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet18.maxpool = nn.Identity()  # remove maxpool since conv1 no longer aggressively downsamples

# train for 10 epochs, batch size 64 (already set in train_loader/test_loader), lr=0.001
results_resnet = train_model(
    resnet18, train_loader, test_loader,
    num_epochs=10, lr=0.001, model_name="ResNet18_scratch"
)


Training: ResNet18_scratch
Epoch 1/10 | Train Loss: 3.6792 | Test Acc: 20.59% | Time: 51.7s
Epoch 2/10 | Train Loss: 2.8689 | Test Acc: 29.33% | Time: 47.8s
Epoch 3/10 | Train Loss: 2.3889 | Test Acc: 40.20% | Time: 48.5s
Epoch 4/10 | Train Loss: 2.0576 | Test Acc: 46.73% | Time: 49.4s
Epoch 5/10 | Train Loss: 1.8134 | Test Acc: 49.41% | Time: 48.4s
Epoch 6/10 | Train Loss: 1.6175 | Test Acc: 53.26% | Time: 48.3s
Epoch 7/10 | Train Loss: 1.4585 | Test Acc: 54.84% | Time: 50.8s
Epoch 8/10 | Train Loss: 1.3237 | Test Acc: 58.32% | Time: 48.3s
Epoch 9/10 | Train Loss: 1.2011 | Test Acc: 56.46% | Time: 48.3s
Epoch 10/10 | Train Loss: 1.0982 | Test Acc: 59.80% | Time: 49.5s

Final Test Accuracy: 59.80%
Total Training Time: 491.1s (8.2 min)
Number of Parameters: 11,220,132


In [ ]:
# 11: FLOPs for ResNet-18

# rebuild ResNet-18 fresh with the same CIFAR-adapted architecture as 10
# just for FLOPs tracing
resnet_for_flops = models.resnet18(weights=None)
resnet_for_flops.fc = nn.Linear(resnet_for_flops.fc.in_features, 100)
resnet_for_flops.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
resnet_for_flops.maxpool = nn.Identity()
resnet_for_flops = resnet_for_flops.to(device)

resnet_summary = summary(
    resnet_for_flops,
    input_size=(1, 3, 32, 32),
    verbose=0,
    col_names=["num_params", "mult_adds"],
)

resnet_flops = resnet_summary.total_mult_adds
resnet_params = resnet_summary.total_params

print(f"ResNet18_scratch:")
print(f"  Params: {resnet_params:,}")
print(f"  FLOPs (mult-adds) per forward pass: {resnet_flops:,}")

flops_results["ResNet18_scratch"] = {
    "total_params": resnet_params,
    "flops_per_forward_pass": resnet_flops,
}

del resnet_for_flops
torch.cuda.empty_cache()

ResNet18_scratch:
  Params: 11,220,132
  FLOPs (mult-adds) per forward pass: 555,478,500


In [ ]:
# 12 (updated): results

summary_rows = []
for res in all_results:
    model_name = res["model_name"]
    flops = flops_results[model_name]["flops_per_forward_pass"]

    summary_rows.append({
        "Model": model_name,
        "Final Test Acc (%)": round(res["final_test_accuracy"], 2),
        "Num Params": f"{res['num_params']:,}",
        "FLOPs/forward pass": f"{flops:,}",
        "Avg Epoch Time (s)": round(res["avg_epoch_time"], 1),
        "Total Train Time (min)": round(res["total_training_time"] / 60, 1),
    })

results_df = pd.DataFrame(summary_rows)
results_df = results_df.sort_values("Final Test Acc (%)", ascending=False).reset_index(drop=True)

print(results_df.to_string(index=False))

results_df.to_csv("problem1_results_summary.csv", index=False)
print("\nSaved to problem1_results_summary.csv")

                Model  Final Test Acc (%) Num Params FLOPs/forward pass  Avg Epoch Time (s)  Total Train Time (min)
     ResNet18_scratch               59.80 11,220,132        555,478,500                49.1                     8.2
    Config_A_baseline               34.23  3,214,692          2,935,396                36.9                     6.2
      Config_D_deeper               14.33  6,373,732          5,041,764                70.1                    11.7
Config_C_larger_patch               11.74  3,239,268          2,923,108                29.9                     5.0
       Config_B_wider                7.17 12,720,740         10,064,996               105.8                    17.6

Saved to problem1_results_summary.csv


In [ ]:
#problem 2

In [ ]:
#13: install hugging face transformers library

!pip install -q transformers

import transformers
print(f"Transformers version: {transformers.__version__}")

Transformers version: 5.12.1


In [ ]:
# disable hf_transfer since it didn't help
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

# download Swin-Tiny files directly via wget
!mkdir -p ./swin_tiny_local
!wget -q --show-progress https://huggingface.co/microsoft/swin-tiny-patch4-window7-224/resolve/main/config.json -O ./swin_tiny_local/config.json
!wget -q --show-progress https://huggingface.co/microsoft/swin-tiny-patch4-window7-224/resolve/main/model.safetensors -O ./swin_tiny_local/model.safetensors
!wget -q --show-progress https://huggingface.co/microsoft/swin-tiny-patch4-window7-224/resolve/main/preprocessor_config.json -O ./swin_tiny_local/preprocessor_config.json

print("Download complete")
!ls -lh ./swin_tiny_local/

./swin_tiny_local/c 100%[===================>]  70.13K  --.-KB/s    in 0.01s   
./swin_tiny_local/m 100%[===================>] 108.16M   283MB/s    in 0.4s    
./swin_tiny_local/p 100%[===================>]     255  --.-KB/s    in 0s      
Download complete
total 109M
-rw-r--r-- 1 root root  71K Jul 15 00:41 config.json
-rw-r--r-- 1 root root 109M Jul 15 00:41 model.safetensors
-rw-r--r-- 1 root root  255 Jul 15 00:41 preprocessor_config.json


In [ ]:
#14: load Swin-Tiny from the locally downloaded files

from transformers import SwinForImageClassification

swin_tiny = SwinForImageClassification.from_pretrained(
    "./swin_tiny_local",
    num_labels=100,                  # adapt for CIFAR-100's 100 classes
    ignore_mismatched_sizes=True,    # allows replacing the original 1000-class head
)

# freeze the backbone
# only the head will be trained during fine-tuning
for name, param in swin_tiny.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

# confirm only the classifier head has trainable params
trainable_params = sum(p.numel() for p in swin_tiny.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in swin_tiny.parameters())
print(f"Swin-Tiny loaded. Trainable params: {trainable_params:,} / {total_params:,} total")

[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: ./swin_tiny_local
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Swin-Tiny loaded. Trainable params: 76,900 / 27,596,254 total


In [ ]:
#15: load Swin-Small, adapt head, freeze backbone

!mkdir -p ./swin_small_local

# weights file for Swin-Small (pytorch_model.bin, not safetensors)
!wget -q --show-progress https://huggingface.co/microsoft/swin-small-patch4-window7-224/resolve/main/config.json -O ./swin_small_local/config.json
!wget -q --show-progress https://huggingface.co/microsoft/swin-small-patch4-window7-224/resolve/main/pytorch_model.bin -O ./swin_small_local/pytorch_model.bin
!wget -q --show-progress https://huggingface.co/microsoft/swin-small-patch4-window7-224/resolve/main/preprocessor_config.json -O ./swin_small_local/preprocessor_config.json

print("Download complete")
!ls -lh ./swin_small_local/

# Load Swin-Small from local files
swin_small = SwinForImageClassification.from_pretrained(
    "./swin_small_local",
    num_labels=100,
    ignore_mismatched_sizes=True,
)

# freeze backbone
for name, param in swin_small.named_parameters():
    if "classifier" not in name:
        param.requires_grad = False

trainable_params = sum(p.numel() for p in swin_small.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in swin_small.parameters())
print(f"Swin-Small loaded. Trainable params: {trainable_params:,} / {total_params:,} total")

./swin_small_local/ 100%[===================>]  70.13K  --.-KB/s    in 0.01s   
./swin_small_local/ 100%[===================>] 189.84M   207MB/s    in 0.9s    
./swin_small_local/ 100%[===================>]     255  --.-KB/s    in 0s      
Download complete
total 190M
-rw-r--r-- 1 root root  71K Jul 15 00:44 config.json
-rw-r--r-- 1 root root  255 Jul 15 00:44 preprocessor_config.json
-rw-r--r-- 1 root root 190M Jul 15 00:44 pytorch_model.bin


[transformers] You passed `num_labels=100` which is incompatible to the `id2label` map of length `1000`.


Loading weights:   0%|          | 0/425 [00:00<?, ?it/s]

[transformers] SwinForImageClassification LOAD REPORT from: ./swin_small_local
Key               | Status   |                                                                                            
------------------+----------+--------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 768]) vs model:torch.Size([100, 768])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([100])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


Swin-Small loaded. Trainable params: 76,900 / 48,914,158 total


In [ ]:
# swin data loaders (224x224 resize for pretrained/scratch Swin models)


from torchvision import transforms as T

swin_train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

swin_test_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

swin_train_dataset = torchvision.datasets.CIFAR100(
    root=DRIVE_DATA_PATH, train=True, download=True, transform=swin_train_transform
)
swin_test_dataset = torchvision.datasets.CIFAR100(
    root=DRIVE_DATA_PATH, train=False, download=True, transform=swin_test_transform
)

SWIN_BATCH_SIZE = 32

swin_train_loader = DataLoader(
    swin_train_dataset, batch_size=SWIN_BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
swin_test_loader = DataLoader(
    swin_test_dataset, batch_size=SWIN_BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

print(f"Swin train samples: {len(swin_train_dataset)}, test samples: {len(swin_test_dataset)}")

Swin train samples: 50000, test samples: 10000


In [ ]:
# 16: fine-tune Swin-Tiny (5 epochs, batch 32, lr 2e-5)

import torch.optim as optim

def train_swin_model(model, train_loader, test_loader, num_epochs=5, lr=2e-5, model_name="model"):
    """
    Fine-tunes a Hugging Face Swin model. Similar structure to train_model() from Cell 4,
    but adapted for HF's output format (model returns an object with .logits, not a raw tensor).
    """
    model = model.to(device)

    # only optimize parameters that require gradients
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []
    epoch_times = []

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # training phase
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(pixel_values=images)   # HF models take pixel_values as the input kwarg
            loss = criterion(outputs.logits, labels)  # HF output object has .logits attribute
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        avg_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        # evaluation phase
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(pixel_values=images)
                _, predicted = torch.max(outputs.logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total
        test_accuracies.append(test_accuracy)

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Test Acc: {test_accuracy:.2f}% | "
              f"Time: {epoch_time:.1f}s")

    total_training_time = time.time() - total_start_time

    results = {
        "model_name": model_name,
        "train_losses": train_losses,
        "test_accuracies": test_accuracies,
        "final_test_accuracy": test_accuracies[-1],
        "epoch_times": epoch_times,
        "avg_epoch_time": sum(epoch_times) / len(epoch_times),
        "total_training_time": total_training_time,
        "num_params": sum(p.numel() for p in model.parameters()),
    }

    print(f"\nFinal Test Accuracy: {results['final_test_accuracy']:.2f}%")
    print(f"Total Training Time: {total_training_time:.1f}s ({total_training_time/60:.1f} min)")

    return results


# fine-tune Swin-Tiny using the Swin-specific 224x224 loaders
results_swin_tiny = train_swin_model(
    swin_tiny, swin_train_loader, swin_test_loader,
    num_epochs=5, lr=2e-5, model_name="Swin_Tiny_pretrained"
)


Training: Swin_Tiny_pretrained
Epoch 1/5 | Train Loss: 4.1064 | Test Acc: 44.93% | Time: 295.2s
Epoch 2/5 | Train Loss: 3.1932 | Test Acc: 56.94% | Time: 293.8s
Epoch 3/5 | Train Loss: 2.5393 | Test Acc: 61.22% | Time: 294.0s
Epoch 4/5 | Train Loss: 2.1014 | Test Acc: 63.37% | Time: 293.8s
Epoch 5/5 | Train Loss: 1.8094 | Test Acc: 64.88% | Time: 293.8s

Final Test Accuracy: 64.88%
Total Training Time: 1470.7s (24.5 min)


In [ ]:
# 17: fine-tune Swin-Small (5 epochs, batch 32, lr 2e-5)

results_swin_small = train_swin_model(
    swin_small, swin_train_loader, swin_test_loader,
    num_epochs=5, lr=2e-5, model_name="Swin_Small_pretrained"
)


Training: Swin_Small_pretrained
Epoch 1/5 | Train Loss: 4.0286 | Test Acc: 52.07% | Time: 526.9s
Epoch 2/5 | Train Loss: 3.0156 | Test Acc: 62.24% | Time: 524.8s
Epoch 3/5 | Train Loss: 2.3140 | Test Acc: 65.69% | Time: 526.0s
Epoch 4/5 | Train Loss: 1.8673 | Test Acc: 67.59% | Time: 526.3s
Epoch 5/5 | Train Loss: 1.5898 | Test Acc: 68.86% | Time: 528.4s

Final Test Accuracy: 68.86%
Total Training Time: 2632.5s (43.9 min)


In [ ]:
# 18: build swin transformer from scratch

from transformers import SwinConfig, SwinForImageClassification

#  SwinConfig matching Swin-Tiny's architecture
swin_scratch_config = SwinConfig(
    image_size=224,
    patch_size=4,
    num_channels=3,
    embed_dim=96,                    # same as Swin-Tiny's base embedding dim
    depths=[2, 2, 6, 2],              # same stage depths as Swin-Tiny
    num_heads=[3, 6, 12, 24],         # same head counts as Swin-Tiny
    window_size=7,
    num_labels=100,                   # CIFAR-100 classes - built in from the start
)

swin_scratch = SwinForImageClassification(swin_scratch_config)  # random init, no pretrained weights loaded


# trains from scratch
total_params = sum(p.numel() for p in swin_scratch.parameters())
trainable_params = sum(p.numel() for p in swin_scratch.parameters() if p.requires_grad)
print(f"Swin-scratch built. Trainable params: {trainable_params:,} / {total_params:,} total")

# verify output shape with a dummy batch
test_input = torch.randn(2, 3, 224, 224).to(device)
swin_scratch = swin_scratch.to(device)
test_output = swin_scratch(pixel_values=test_input)
print(f"Test output shape: {test_output.logits.shape}")  # expect [2, 100]
del test_input, test_output
torch.cuda.empty_cache()

Swin-scratch built. Trainable params: 27,596,254 / 27,596,254 total
Test output shape: torch.Size([2, 100])


In [ ]:
import torch.optim as optim

def train_swin_model(model, train_loader, test_loader, num_epochs=5, lr=2e-5, model_name="model"):
    """
    Fine-tunes/trains a Hugging Face Swin model. Similar structure to train_model()
    from Problem 1, but adapted for HF's output format (model returns an object
    with .logits, not a raw tensor).
    """
    model = model.to(device)

    # only optimize parameters that require gradients
    # (pretrained models head - for scratch everything)
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []
    epoch_times = []

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")

    total_start_time = time.time()

    for epoch in range(num_epochs):
        epoch_start_time = time.time()

        # training phase
        model.train()
        running_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(pixel_values=images)          # HF models take pixel_values as input kwarg
            loss = criterion(outputs.logits, labels)       # HF output object has .logits attribute
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)

        avg_train_loss = running_loss / len(train_loader.dataset)
        train_losses.append(avg_train_loss)

        # evaluation phase
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(pixel_values=images)
                _, predicted = torch.max(outputs.logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        test_accuracy = 100 * correct / total
        test_accuracies.append(test_accuracy)

        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)

        print(f"Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss: {avg_train_loss:.4f} | "
              f"Test Acc: {test_accuracy:.2f}% | "
              f"Time: {epoch_time:.1f}s")

    total_training_time = time.time() - total_start_time

    # package everything into a results dict for the final comparison table
    results = {
        "model_name": model_name,
        "train_losses": train_losses,
        "test_accuracies": test_accuracies,
        "final_test_accuracy": test_accuracies[-1],
        "epoch_times": epoch_times,
        "avg_epoch_time": sum(epoch_times) / len(epoch_times),
        "total_training_time": total_training_time,
        "num_params": sum(p.numel() for p in model.parameters()),
    }

    print(f"\nFinal Test Accuracy: {results['final_test_accuracy']:.2f}%")
    print(f"Total Training Time: {total_training_time:.1f}s ({total_training_time/60:.1f} min)")

    return results

In [ ]:
#19: train swin-from-scratch (5 epochs, batch 32, lr 0.001)

results_swin_scratch = train_swin_model(
    swin_scratch, swin_train_loader, swin_test_loader,
    num_epochs=5, lr=0.001, model_name="Swin_scratch"
)


Training: Swin_scratch
Epoch 1/5 | Train Loss: 4.6485 | Test Acc: 1.00% | Time: 782.0s
Epoch 2/5 | Train Loss: 4.6191 | Test Acc: 1.00% | Time: 812.7s
Epoch 3/5 | Train Loss: 4.6132 | Test Acc: 1.00% | Time: 807.0s
Epoch 4/5 | Train Loss: 4.6111 | Test Acc: 1.00% | Time: 808.6s
Epoch 5/5 | Train Loss: 4.6091 | Test Acc: 1.00% | Time: 808.8s

Final Test Accuracy: 1.00%
Total Training Time: 4019.1s (67.0 min)
